# TP-2: CleaningData - Applying OOP and Design Patterns


## Exercise 1: Build the CSVReader Class (OOP Foundation)
**Objective:**
* Reinforce OOP fundamentals: classes, constructors, methods, encapsulation.
* Learn to design reusable data classes for DS projects.

**Concept:**
A CSVReader class should encapsulate all functionality related to reading CSV files.

**Instructions:**
1.	Create a class CSVReader.
2.	Define an __init__ method that takes file_path.
3.	Add a read() method that loads the CSV file using pandas.
4.	Add a preview(n) method that prints the top n rows.

In [1]:
import pandas as pd

class CSVReader:
    def __init__(self, file_path: str):
        self.file_path = file_path
        self.df = None

    def read(self) -> pd.DataFrame:
        try:
            self.df = pd.read_csv(self.file_path)
            print(f"Successfully loaded file from: {self.file_path}")
            return self.df
        except Exception as e:
            print(f"Error loading file: {e}")
            return None
        
    def preview(self, n: int) -> pd.DataFrame:
        if self.df is not None:
            print(f"Previewing top {n} rows:")
            return self.df.head(n)
        else:
            print("No data loaded yet. Please run the read() method first.")
            return None

In [2]:
read = CSVReader("sample_data (2).csv")
read.read()

Successfully loaded file from: sample_data (2).csv


,id,name,age,height_cm,weight_kg,city,score
0,1,Alice,29.0,165.0,68.0,New York,85.0
1,2,Bob,NaN,172.0,NaN,Los Angeles,90.0
2,3,Charlie,35.0,168.0,72.0,Chicago,NaN
3,4,David,NaN,NaN,80.0,Houston,75.0
4,5,Eva,27.0,160.0,55.0,New York,88.0
5,6,Frank,30.0,175.0,85.0,Los Angeles,NaN
6,7,Grace,NaN,162.0,60.0,Chicago,92.0


In [3]:
read.preview(5)

Previewing top 5 rows:


,id,name,age,height_cm,weight_kg,city,score
0,1,Alice,29.0,165.0,68.0,New York,85.0
1,2,Bob,NaN,172.0,NaN,Los Angeles,90.0
2,3,Charlie,35.0,168.0,72.0,Chicago,NaN
3,4,David,NaN,NaN,80.0,Houston,75.0
4,5,Eva,27.0,160.0,55.0,New York,88.0


## Exercise 2: Apply Pattern for Missing Values Handling
**Objective:**
* Learn the Strategy Pattern - one interface, many interchangeable behaviors.
* Practice inheritance and polymorphism.

**Concept:**
We often need different strategies to handle missing data (drop, fill mean, fill mode, etc.).
The pattern allows flexible switching between methods without changing the main code.

**Instructions:**
1.	Create an abstract class MissingValueStrategy with a method handle(df).
2.	Create subclasses:
    *	DropMissing
    *	FillMean
    *	FillMode
3.	Create a DataCleaner class that applies the chosen strategy.


In [4]:
from abc import ABC, abstractmethod
import pandas as pd

# 1. Abstract Base Class (Strategy Interface)
class MissingValueStrategy(ABC):
    @abstractmethod
    def handle(self, df: pd.DataFrame) -> pd.DataFrame:
        """Abstract method that all strategies must implement."""
        pass

# 2. Concrete Strategy: Drop rows with missing values
class DropMissing(MissingValueStrategy):
    def handle(self, df: pd.DataFrame) -> pd.DataFrame:
        print("-> Applying Strategy: Dropping missing values.")
        return df.dropna()

# 3. Concrete Strategy: Fill missing values with the mean (numeric columns)
class FillMean(MissingValueStrategy):
    def handle(self, df: pd.DataFrame) -> pd.DataFrame:
        print("-> Applying Strategy: Filling missing values with mean.")
        # Fills numeric columns with their mean value
        return df.fillna(df.mean(numeric_only=True))

# 4. Concrete Strategy: Fill missing values with the mode (most frequent value)
class FillMode(MissingValueStrategy):
    def handle(self, df: pd.DataFrame) -> pd.DataFrame:
        print("-> Applying Strategy: Filling missing values with mode.")
        return df.fillna(df.mode().iloc[0])

# 5. DataCleaner Context Class
class DataCleaner:
    def __init__(self, strategy: MissingValueStrategy):
        """Initializes the DataCleaner with a chosen strategy."""
        self.strategy = strategy

    def set_strategy(self, strategy: MissingValueStrategy):
        """Allows changing the strategy dynamically at runtime."""
        self.strategy = strategy

    def clean_data(self, df: pd.DataFrame) -> pd.DataFrame:
        """Executes the current strategy's handle method on the DataFrame."""
        if self.strategy is None:
            raise ValueError("No missing value strategy has been set!")
        return self.strategy.handle(df)

In [5]:
df = read.read()
df

Successfully loaded file from: sample_data (2).csv


,id,name,age,height_cm,weight_kg,city,score
0,1,Alice,29.0,165.0,68.0,New York,85.0
1,2,Bob,NaN,172.0,NaN,Los Angeles,90.0
2,3,Charlie,35.0,168.0,72.0,Chicago,NaN
3,4,David,NaN,NaN,80.0,Houston,75.0
4,5,Eva,27.0,160.0,55.0,New York,88.0
5,6,Frank,30.0,175.0,85.0,Los Angeles,NaN
6,7,Grace,NaN,162.0,60.0,Chicago,92.0


In [6]:
cleaner = DataCleaner(FillMean())
result = cleaner.clean_data(df)
print(result)

-> Applying Strategy: Filling missing values with mean.
   id     name    age  height_cm  weight_kg         city  score
0   1    Alice  29.00      165.0       68.0     New York   85.0
1   2      Bob  30.25      172.0       70.0  Los Angeles   90.0
2   3  Charlie  35.00      168.0       72.0      Chicago   86.0
3   4    David  30.25      167.0       80.0      Houston   75.0
4   5      Eva  27.00      160.0       55.0     New York   88.0
5   6    Frank  30.00      175.0       85.0  Los Angeles   86.0
6   7    Grace  30.25      162.0       60.0      Chicago   92.0


In [7]:
cleaner = DataCleaner(FillMode())
result = cleaner.clean_data(df)
print(result)

-> Applying Strategy: Filling missing values with mode.
   id     name   age  height_cm  weight_kg         city  score
0   1    Alice  29.0      165.0       68.0     New York   85.0
1   2      Bob  27.0      172.0       55.0  Los Angeles   90.0
2   3  Charlie  35.0      168.0       72.0      Chicago   75.0
3   4    David  27.0      160.0       80.0      Houston   75.0
4   5      Eva  27.0      160.0       55.0     New York   88.0
5   6    Frank  30.0      175.0       85.0  Los Angeles   75.0
6   7    Grace  27.0      162.0       60.0      Chicago   92.0


## Exercise 3: Add Decorators for Logging and Timing
**Objective:**
* Learn the Decorator Pattern to add reusable functionality (logging, timing).
* Understand how decorators support the “Open/Closed Principle”.

**Concept:**
Decorators wrap functions to extend their behavior without altering the original code.

**Instructions:**
1.	Create two decorators:
    *	@log_action → logs when a method starts and finishes.
    *	@log_time → calculates and prints execution time.
2.	Apply them to methods in CSVReader or DataCleaner.


In [8]:
import time
from functools import wraps

def log_action(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        print(f"[LOG] Starting action: '{func.__name__}'")
        result = func(*args, **kwargs)
        print(f"[LOG] Finished action: '{func.__name__}'")
        return result
    return wrapper

def log_time(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        start_time = time.time()
        result = func(*args, **kwargs)
        end_time = time.time()
        elapsed_time = end_time - start_time
        print(f"[TIME] Action '{func.__name__}' took {elapsed_time:.4f} seconds to execute.")
        return result
    return wrapper

class DataCleaner:
    def __init__(self, strategy: MissingValueStrategy):
        self.strategy = strategy

    def set_strategy(self, strategy: MissingValueStrategy):
        self.strategy = strategy

    @log_action
    @log_time
    def clean_data(self, df: pd.DataFrame) -> pd.DataFrame:
        """Executes the chosen missing value strategy with logging and timing."""
        if self.strategy is None:
            raise ValueError("No missing value strategy has been set!")
        return self.strategy.handle(df)

## Exercise 4: Implement Factory Pattern for Data Transformations
**Objective:**
* Practice Factory Pattern for scalable creation of transformation objects.
* Apply abstraction and composition.

**Concept:**
Instead of manually creating objects, use a factory that decides what transformation to apply.

**Instructions:**
1.	Create an abstract class DataTransform with a method apply(df).
2.	Implement subclasses:
    *	NormalizeColumns
    *	RemoveDuplicates
    *	StandardizeText
3.	Create a TransformFactory that returns transformation objects based on string input.


In [9]:
from abc import ABC, abstractmethod
import pandas as pd

class DataTransform(ABC):
    @abstractmethod
    def apply(self, df: pd.DataFrame) -> pd.DataFrame:
        """Abstract method that all transformations must implement."""
        pass

class RemoveDuplicates(DataTransform):
    def apply(self, df: pd.DataFrame) -> pd.DataFrame:
        print("-> Applying Transformation: Removing duplicate rows.")
        return df.drop_duplicates()

class StandardizeText(DataTransform):
    def apply(self, df: pd.DataFrame) -> pd.DataFrame:
        print("-> Applying Transformation: Standardizing text columns.")
        df = df.copy()
        for col in df.select_dtypes(include=['object']):
            df[col] = df[col].astype(str).str.strip().str.lower()
        return df

class NormalizeColumns(DataTransform):
    def apply(self, df: pd.DataFrame) -> pd.DataFrame:
        print("-> Applying Transformation: Normalizing numeric columns.")
        df = df.copy()
        numeric_cols = df.select_dtypes(include=['number']).columns
        for col in numeric_cols:
            min_val = df[col].min()
            max_val = df[col].max()
            if max_val - min_val != 0:
                df[col] = (df[col] - min_val) / (max_val - min_val)
        return df

class TransformFactory:
    @staticmethod
    def create_transformer(transform_type: str) -> DataTransform:
        """Returns a specific transformation object based on string input."""
        transform_type = transform_type.lower().strip()
        
        if transform_type == "remove_duplicates":
            return RemoveDuplicates()
        elif transform_type == "standardize_text":
            return StandardizeText()
        elif transform_type == "normalize":
            return NormalizeColumns()
        else:
            raise ValueError(f"Unknown transformation type: '{transform_type}'")

In [11]:
df = read.read()
cleaner = DataCleaner(FillMean())
result = cleaner.clean_data(df)

print(result)

Successfully loaded file from: sample_data (2).csv
[LOG] Starting action: 'clean_data'
-> Applying Strategy: Filling missing values with mean.
[TIME] Action 'clean_data' took 0.0010 seconds to execute.
[LOG] Finished action: 'clean_data'
   id     name    age  height_cm  weight_kg         city  score
0   1    Alice  29.00      165.0       68.0     New York   85.0
1   2      Bob  30.25      172.0       70.0  Los Angeles   90.0
2   3  Charlie  35.00      168.0       72.0      Chicago   86.0
3   4    David  30.25      167.0       80.0      Houston   75.0
4   5      Eva  27.00      160.0       55.0     New York   88.0
5   6    Frank  30.00      175.0       85.0  Los Angeles   86.0
6   7    Grace  30.25      162.0       60.0      Chicago   92.0


## Exercise 5: Build a Full Cleaning Pipeline (Template Method Pattern)
**Objective:**
* Combine all previous concepts into one cleaning pipeline.
* Use the Template Method Pattern to define a consistent workflow.

**Concept:**
The Template Method defines the skeleton of a process and lets subclasses override specific steps.

**Instructions:**
1.	Create an abstract class DataPipeline with a run() method defining steps:
    * load()
    * clean()
    * transform()
    * save()
2.	Create CSVDataPipeline that implements each step using previous classes.



In [10]:
from abc import ABC, abstractmethod
import pandas as pd

# 1. Abstract Base Class defining the Template Method workflow
class DataPipeline(ABC):
    
    def run(self, file_path: str, output_path: str, missing_strategy, transform_types: list) -> pd.DataFrame:
        """
        Template method that defines the skeleton of the data cleaning process.
        Subclasses implement the specific details of each step.
        """
        print("=== Pipeline Execution Started ===")
        df = self.load(file_path)
        df = self.clean(df, missing_strategy)
        df = self.transform(df, transform_types)
        self.save(df, output_path)
        print("=== Pipeline Execution Completed ===")
        return df

    @abstractmethod
    def load(self, file_path: str) -> pd.DataFrame:
        pass

    @abstractmethod
    def clean(self, df: pd.DataFrame, strategy) -> pd.DataFrame:
        pass

    @abstractmethod
    def transform(self, df: pd.DataFrame, transform_types: list) -> pd.DataFrame:
        pass

    @abstractmethod
    def save(self, df: pd.DataFrame, output_path: str) -> None:
        pass

# 2. Concrete Implementation of the Pipeline
class CSVDataPipeline(DataPipeline):
    
    def load(self, file_path: str) -> pd.DataFrame:
        print("[Step 1/4: Load]")
        reader = CSVReader(file_path)
        return reader.read()

    def clean(self, df: pd.DataFrame, strategy) -> pd.DataFrame:
        print("[Step 2/4: Clean Missing Values]")
        cleaner = DataCleaner(strategy)
        return cleaner.clean_data(df)

    def transform(self, df: pd.DataFrame, transform_types: list) -> pd.DataFrame:
        print("[Step 3/4: Apply Transformations]")
        for t_type in transform_types:
            transformer = TransformFactory.create_transformer(t_type)
            df = transformer.apply(df)
        return df

    def save(self, df: pd.DataFrame, output_path: str) -> None:
        print(f"[Step 4/4: Save] Saving cleaned data to {output_path}")
        df.to_csv(output_path, index=False)

In [12]:
# Instantiate the pipeline
pipeline = CSVDataPipeline()

# Initialize parameters for the pipeline
input_file = "sample_data (2).csv"
output_file = "cleaned_data.csv"
chosen_strategy = FillMean()  # From Exercise 2
transformations = ["remove_duplicates", "standardize_text"]  # From Exercise 4

# Run the pipeline!
final_df = pipeline.run(
    file_path=input_file,
    output_path=output_file,
    missing_strategy=chosen_strategy,
    transform_types=transformations
)

=== Pipeline Execution Started ===
[Step 1/4: Load]
Successfully loaded file from: sample_data (2).csv
[Step 2/4: Clean Missing Values]
[LOG] Starting action: 'clean_data'
-> Applying Strategy: Filling missing values with mean.
[TIME] Action 'clean_data' took 0.0009 seconds to execute.
[LOG] Finished action: 'clean_data'
[Step 3/4: Apply Transformations]
-> Applying Transformation: Removing duplicate rows.
-> Applying Transformation: Standardizing text columns.
[Step 4/4: Save] Saving cleaned data to cleaned_data.csv
=== Pipeline Execution Completed ===
